# RECOMOVIE.AI — Etapa 3: Validação robusta

Este notebook valida o recomendador como **problema de clusterização**, mantendo MiniBatch K-Means e similaridade de cosseno.

In [ ]:
from pathlib import Path
import json, pandas as pd
from src.config import default_config
from src.data import MovieRepository
from src.model import ClusterModel

BASE=Path.cwd(); cfg=default_config(BASE)
repo=MovieRepository(cfg.data_path,cfg.movies_path)
model=ClusterModel(cfg.n_clusters,cfg.genre_weight,cfg.batch_size,cfg.epochs,cfg.random_state)
X=model.make_features(repo.df,repo.genre_columns,fit=True)
print(f"Filmes: {len(repo.df):,}")
print(f"Características: {X.shape[1]}")
print(f"Clusters oficiais: {cfg.n_clusters}")

## 1. Métricas internas de qualidade

Avaliamos Silhouette, Davies-Bouldin e Calinski-Harabasz em uma amostra de 10 mil filmes para manter a análise executável localmente.

In [ ]:
exec(open("validacao_robusta.py", encoding="utf-8").read())

## 2. Sensibilidade ao número de clusters

Comparamos diferentes valores de `k` e observamos como a qualidade dos agrupamentos muda. O gráfico gerado em `reports/sensibilidade_k.png` deve ser discutido junto com o objetivo do recomendador.

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv("reports/sensibilidade_k.csv"))

## 3. Estabilidade com diferentes seeds

O ARI compara os agrupamentos obtidos com diferentes inicializações. Quanto mais próximo de 1, maior a estabilidade entre execuções.

In [ ]:
metrics=json.load(open("reports/metricas.json",encoding="utf-8")); print(metrics["estabilidade_seeds"])

## 4. Visualização em 2D via PCA

A projeção em duas componentes ajuda a interpretar se existem regiões separadas ou sobrepostas no espaço de características. Ela é visualização e não substitui as métricas quantitativas.

In [ ]:
from IPython.display import Image, display
display(Image(filename="reports/clusters_pca.png"))

## 5. Discussão crítica e limitações

O sistema não é um classificador nem um regressor. Ele encontra grupos de filmes a partir dos gêneros e da média de avaliação e usa similaridade de cosseno para ordenar candidatos dentro do grupo.

As principais limitações são: dependência das características disponíveis no dataset; ausência de histórico individual do usuário; dificuldade para representar filmes muito raros; e possibilidade de agrupamentos menos estáveis ou menos úteis para alguns valores de `k`. Em uso real, o modelo deveria ser reavaliado com dados novos e com sinais de preferência dos próprios usuários.